In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Section 3.4 — Decision Trees and Random Forests

**Objectives**

By the end of this section you will be able to:

- Build and visualise a Decision Tree classifier.
- Explain overfitting and use `max_depth` to control tree complexity.
- Train a Random Forest and understand how ensemble averaging reduces variance.
- Apply cross-validation to produce a reliable performance estimate.

> **Cooking analogy:** A **Decision Tree** is a **flowchart pinned to the
> kitchen wall**: *"Is the meat temperature above 75 °C? Yes → rest 5 minutes.
> No → return to the grill."* Each yes/no question narrows the options until you
> reach a final decision. A **Random Forest** is what happens when you ask
> **100 chefs** to each follow a slightly different flowchart and then vote on
> the best answer. The majority verdict is almost always better than any single
> chef's judgement alone.

Decision Trees and Random Forests are among the most popular algorithms in
business ML because they are **interpretable** and handle mixed data types well.

### Decision Trees

A Decision Tree splits data into groups by asking a series of yes/no questions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

# Re-use churn dataset
np.random.seed(42)
n = 800
df_churn = pd.DataFrame({
    "Tenure_Months" : np.random.randint(1, 72, n),
    "Monthly_Spend" : np.random.normal(70, 25, n).clip(20, 200).round(2),
    "Support_Calls" : np.random.poisson(1.5, n),
    "Satisfaction"  : np.random.uniform(1, 10, n).round(1),
    "Num_Products"  : np.random.randint(1, 6, n),
})
df_churn["Churned"] = (
    (df_churn["Satisfaction"] < 4.5) |
    (df_churn["Support_Calls"] > 4)
).astype(int)

X = df_churn.drop(columns="Churned")
y = df_churn["Churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
# Train a shallow decision tree (max 3 levels — readable)
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
print("Decision Tree Accuracy:", f"{accuracy_score(y_test, y_pred_dt):.2%}")

In [ ]:
# Visualise the tree
fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(dt,
          feature_names=X.columns,
          class_names=["Stayed", "Churned"],
          filled=True, rounded=True, fontsize=10, ax=ax)
ax.set_title("Decision Tree (max_depth=3)", fontsize=14)
plt.tight_layout()
plt.show()

### Overfitting in Decision Trees

> **Cooking analogy:** A tree that grows too deep is like a chef who **memorises
> one customer's exact order** down to every quirk, instead of learning to cook
> well in general. The next customer gets a terrible meal. `max_depth` is the
> rule that says *"stop refining after three decision levels"* — keeping the
> recipe general enough to work for everyone.

In [ ]:
depths     = range(1, 16)
train_acc  = []
test_acc   = []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, clf.predict(X_train)))
    test_acc.append(accuracy_score(y_test,  clf.predict(X_test)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depths, train_acc, "b-o", label="Train Accuracy")
ax.plot(depths, test_acc,  "r-o", label="Test Accuracy")
ax.set_xlabel("Max Tree Depth")
ax.set_ylabel("Accuracy")
ax.set_title("Decision Tree: Accuracy vs Depth")
ax.legend()
ax.axvline(x=3, color="green", linestyle="--", label="Optimal depth ≈ 3")
plt.tight_layout()
plt.show()

### Random Forest — Ensemble Learning

A Random Forest builds **many decision trees** on random data subsets, then
**averages** their predictions — like asking **100 experienced chefs** to each
taste a dish independently and taking their consensus verdict. One chef might be
wrong; 100 chefs in strong agreement are almost certainly right. This reduces
variance without much bias increase.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=6,
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", f"{accuracy_score(y_test, y_pred_rf):.2%}")
print(classification_report(y_test, y_pred_rf, target_names=["Stayed","Churned"]))

In [ ]:
# Feature importance
imp_df = pd.DataFrame({
    "Feature"   : X.columns,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp_df["Feature"], imp_df["Importance"], color="teal")
ax.set_xlabel("Importance")
ax.set_title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation for robust evaluation
cv_scores = cross_val_score(rf, X, y, cv=5, scoring="accuracy")
print(f"5-Fold CV Accuracy: {cv_scores.mean():.2%} ± {cv_scores.std():.2%}")

---

### Student Task 3.4

1. Train a `DecisionTreeClassifier` on the loan approval dataset (`df_loan`) with `max_depth=4`.
2. Visualise the tree and identify the most important splitting feature at the root.
3. Train a `RandomForestClassifier` with 100 trees and compare accuracy with the single tree.
4. Plot feature importance from the Random Forest.
5. Explain in plain language why a Random Forest generally outperforms a single Decision Tree.

In [ ]:
# Your code here

---

### Evaluation Questions 3.4

1. What is the purpose of `max_depth` in a Decision Tree?
   a) Limits the number of features used
   b) Limits how many levels deep the tree can grow, preventing overfitting (correct)
   c) Sets the number of trees in the forest
   d) Controls the learning rate

2. A Random Forest improves over a single Decision Tree by:
   a) Using a more complex mathematical formula
   b) Averaging predictions of many trees trained on random subsets (correct)
   c) Using gradient descent
   d) Selecting only the most important features

3. Feature importance in a Random Forest reflects:
   a) The correlation of each feature with the target
   b) How much each feature reduces impurity across all trees (correct)
   c) The number of times each feature appears in the data
   d) The p-value of each feature

4. Cross-validation helps evaluate a model by:
   a) Training the model multiple times with different hyperparameters
   b) Testing the model on multiple different train/test splits (correct)
   c) Reducing the training dataset size
   d) Automatically tuning the number of trees

5. Which statement about Decision Trees is TRUE?
   a) They always require feature scaling
   b) They cannot handle categorical variables
   c) A very deep tree typically overfits to training data (correct)
   d) They produce a linear decision boundary

---